In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
# Read the CSV file into a pandas DataFrame
df = pd.read_csv("seattle-weather.csv")

# Display the first 5 rows of the DataFrame
display(df.head())

# Print the concise summary of the DataFrame
df.info()

# Display descriptive statistics of the DataFrame
display(df.describe())

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1461 non-null   object 
 1   precipitation  1461 non-null   float64
 2   temp_max       1461 non-null   float64
 3   temp_min       1461 non-null   float64
 4   wind           1461 non-null   float64
 5   weather        1461 non-null   object 
dtypes: float64(4), object(2)
memory usage: 68.6+ KB


,precipitation,temp_max,temp_min,wind
count,1461.000000,1461.000000,1461.000000,1461.000000
mean,3.029432,16.439083,8.234771,3.241136
std,6.680194,7.349758,5.023004,1.437825
min,0.000000,-1.600000,-7.100000,0.400000
25%,0.000000,10.600000,4.400000,2.200000
50%,0.000000,15.600000,8.300000,3.000000
75%,2.800000,22.200000,12.200000,4.000000
max,55.900000,35.600000,18.300000,9.500000


## Data Preprocessing

In [7]:
df['date'] = pd.to_datetime(df['date'])
numerical_features = ['precipitation', 'temp_max', 'temp_min', 'wind']
df_numerical = df[numerical_features]

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np

scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_numerical)

def create_sequences(data, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:(i + sequence_length)])
        y.append(data[i + sequence_length])
    return np.array(X), np.array(y)

sequence_length = 10 # Define the sequence length
X, y = create_sequences(df_scaled, sequence_length)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## Model Creation

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential()
model.add(LSTM(units=50, return_sequences=False, input_shape=(sequence_length, df_numerical.shape[1])))
model.add(Dense(units=df_numerical.shape[1]))

model.summary()

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 50)             │        11,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │           204 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,204 (43.77 KB)

 Trainable params: 11,204 (43.77 KB)

 Non-trainable params: 0 (0.00 B)

## Model compilation


Compile the RNN model by specifying the optimizer, loss function, and metrics.


In [11]:
from tensorflow.keras.optimizers import Adam

model.compile(optimizer=Adam(), loss='mean_squared_error', metrics=['mean_absolute_error'])

## Model training


Train the RNN model using the prepared data, splitting it into training and validation sets.


In [12]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 0.0523 - mean_absolute_error: 0.1614 - val_loss: 0.0153 - val_mean_absolute_error: 0.0918
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0152 - mean_absolute_error: 0.0897 - val_loss: 0.0132 - val_mean_absolute_error: 0.0850
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0143 - mean_absolute_error: 0.0855 - val_loss: 0.0129 - val_mean_absolute_error: 0.0833
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0142 - mean_absolute_error: 0.0854 - val_loss: 0.0127 - val_mean_absolute_error: 0.0804
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0140 - mean_absolute_error: 0.0852 - val_loss: 0.0125 - val_mean_absolute_error: 0.0787
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0137 - mean_absolute_error: 0.0832 - val_loss: 0.0124 - val_mean_absolute_error: 0.0825
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0135 - mean_absolute_error: 0.0827 - val_loss: 0.01

## Model evaluation




In [13]:
evaluation_results = model.evaluate(X_val, y_val)
print("Validation Loss:", evaluation_results[0])
print("Validation Mean Absolute Error:", evaluation_results[1])

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0104 - mean_absolute_error: 0.0722 
Validation Loss: 0.010442573577165604
Validation Mean Absolute Error: 0.07218161225318909




### Data Analysis Key Findings

*   The initial attempt to load "data/GOOG.csv" failed; the analysis proceeded using "seattle-weather.csv".
*   The "seattle-weather.csv" dataset contains 1461 entries and 6 columns, with no missing values.
*   Numerical features were scaled using `MinMaxScaler` and converted into sequences of length 10.
*   The sequential data was split into 80% for training and 20% for validation.
*   An RNN model with an LSTM layer (50 units) and a Dense output layer (4 units) was defined.
*   The model was compiled using the Adam optimizer, Mean Squared Error loss, and Mean Absolute Error metric.
*   The model was trained for 50 epochs, showing decreasing loss and MAE on both training and validation sets.
*   On the validation set, the model achieved a Validation Loss of 0.0108 and a Validation Mean Absolute Error of 0.0731.
*   Predictions were made on the validation set and inverse transformed back to the original scale.



## Predicting the O/P for the Weather column out of Rain, drizzle, sun ,snow

In [14]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
df['weather_encoded'] = label_encoder.fit_transform(df['weather'])

print("Weather mapping:", dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

Weather mapping: {'drizzle': 0, 'fog': 1, 'rain': 2, 'snow': 3, 'sun': 4}


In [ ]:
def create_sequences_classification(data, targets, sequence_length):
    X, y = [], []
    for i in range(len(data) - sequence_length):
        X.append(data[i:(i + sequence_length)])  
        y.append(targets[i + sequence_length])   
    return np.array(X), np.array(y)

In [16]:
numerical_features = ['precipitation', 'temp_max', 'temp_min', 'wind']
df_numerical = df[numerical_features]

In [17]:
scaler = MinMaxScaler()
df_scaled = scaler.fit_transform(df_numerical)

In [18]:
sequence_length = 10
X, y = create_sequences_classification(df_scaled, df['weather_encoded'].values, sequence_length)

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical

In [20]:
y_categorical = to_categorical(y)

In [21]:
X_train, X_val, y_train, y_val = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

In [22]:
model = Sequential()
model.add(LSTM(units=64, return_sequences=False, input_shape=(sequence_length, df_numerical.shape[1])))
model.add(Dropout(0.2))  # Prevents overfitting
model.add(Dense(units=32, activation='relu'))
model.add(Dense(units=len(label_encoder.classes_), activation='softmax'))

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [23]:
model.compile(optimizer='adam', 
              loss='categorical_crossentropy',  # Different loss for classification
              metrics=['accuracy'])

In [24]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,909 (77.77 KB)

 Trainable params: 19,909 (77.77 KB)

 Non-trainable params: 0 (0.00 B)

In [25]:
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val))

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.4388 - loss: 1.3415 - val_accuracy: 0.4536 - val_loss: 1.0667
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4853 - loss: 1.1068 - val_accuracy: 0.6323 - val_loss: 1.0220
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5509 - loss: 1.0672 - val_accuracy: 0.6392 - val_loss: 0.9735
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5664 - loss: 1.0472 - val_accuracy: 0.6392 - val_loss: 0.9672
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5655 - loss: 1.0395 - val_accuracy: 0.6426 - val_loss: 0.9537
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5776 - loss: 1.0376 - val_accuracy: 0.6460 - val_loss: 0.9685
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5733 - loss: 1.0367 - val_accuracy: 0.6460 - val_loss: 0.9553
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5750 - loss: 1.0342 - val_accuracy: 0.6495 - val_loss

In [ ]:
predictions = model.predict(X_val)
predicted_classes = np.argmax(predictions, axis=1)

predicted_weather = label_encoder.inverse_transform(predicted_classes)

print("Predicted weather for tomorrow:", predicted_weather[0])

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
Predicted weather for tomorrow: sun
